# Lab 16-01 · Discrete-Time Systems as Computations

Six parts, each built on `dtsys.py`. **Predict first**, then run. The handout is [README.md](README.md).

In [1]:
import math
import shutil
import subprocess
import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt
import numpy as np
from scipy import signal
import dtsys as ds

## Part 1 · Aliasing is identical samples

*Predict:* the shape of the apparent-frequency curve for inputs from 0 to 3 kHz at $f_s$ = 1 kHz.

In [2]:
n = np.arange(64)
a = np.cos(2 * math.pi * 7e3 * n / 8e3)
b = np.cos(2 * math.pi * 1e3 * n / 8e3)
print("max |7 kHz - 1 kHz| samples at 8 kHz:", np.max(np.abs(a - b)))
f = np.linspace(0, 3000, 3001)
fa = [ds.apparent_freq(x, 1000) for x in f]
fig, ax = plt.subplots(figsize=(6, 2.8))
ax.plot(f, fa); ax.set_xlabel("input frequency (Hz)"); ax.set_ylabel("apparent (Hz)"); ax.grid(True, alpha=0.3)
plt.show()
print("1.2 kHz appears at", ds.apparent_freq(1200, 1000), "Hz")

max |7 kHz - 1 kHz| samples at 8 kHz: 5.040412531798211e-14
1.2 kHz appears at 200.0 Hz


## Part 2 · Convolution as counted MACs

In [3]:
rng = np.random.default_rng(16)
x = rng.normal(size=1000)
h = np.random.default_rng(17).normal(size=64)
y, macs = ds.conv_mac(x, h)
print("matches numpy.convolve:", np.allclose(y, np.convolve(x, h)), "| output length", len(y), "| MACs", macs)

matches numpy.convolve: True | output length 1063 | MACs 64000


## Part 3 · Removing mains hum with moving averages

A 10 Hz "biosignal" with 50 Hz hum and a little noise, sampled at 1 kHz. *Predict:* what survives the 20-point and the 100-point averages?

In [4]:
fs = 1000
n = np.arange(3000)
sig = np.cos(2 * math.pi * 10 * n / fs)
hum = 0.5 * np.cos(2 * math.pi * 50 * n / fs + 0.3)
x = sig + hum + 0.05 * np.random.default_rng(3).normal(size=n.size)


def tone_amp(y, f, start=200):
    A = np.c_[np.cos(2 * math.pi * f * n[start:] / fs), np.sin(2 * math.pi * f * n[start:] / fs)]
    return float(np.hypot(*np.linalg.lstsq(A, y[start:], rcond=None)[0]))


fig, ax = plt.subplots(figsize=(8, 3))
ax.plot(n[:400] / fs, x[:400], color="0.7", label="input")
for N in (20, 100):
    y = signal.lfilter(np.full(N, 1 / N), [1], x)
    ax.plot(n[:400] / fs, y[:400], label=f"N = {N}")
    print(f"N = {N:3d}: 10 Hz amplitude {tone_amp(y, 10):.3f} (closed form {ds.ma_gain(N, 2 * math.pi * 10 / fs):.3f}),"
          f" 50 Hz amplitude {tone_amp(y, 50):.4f}")
ax.set_xlabel("t (s)"); ax.legend(fontsize=8)
plt.show()

N =  20: 10 Hz amplitude 0.934 (closed form 0.936), 50 Hz amplitude 0.0000
N = 100: 10 Hz amplitude 0.000 (closed form 0.000), 50 Hz amplitude 0.0000


## Part 4 · EMA against moving average

*Predict:* with the same white-noise variance gain (1/31), which filter is quieter after settling, and which settles first?

In [5]:
N, alpha = 31, 1 / 16
noise = np.random.default_rng(5).normal(size=400_000)
r_ema = ds.ema(alpha, noise)[1000:].std()
r_ma = signal.lfilter(np.full(N, 1 / N), [1], noise)[1000:].std()
print(f"residual noise std: EMA {r_ema:.4f}, MA {r_ma:.4f}, theory {math.sqrt(1 / 31):.4f}")
s_ema = ds.step([alpha], [1, -(1 - alpha)], 120)
s_ma = ds.step(np.full(N, 1 / N), [1], 120)
print("MA exact from n =", int(np.argmax(np.abs(s_ma - 1) < 1e-12)),
      "| EMA within 1% from n =", int(np.argmax(np.abs(s_ema - 1) <= 0.01)))

residual noise std: EMA 0.1787, MA 0.1787, theory 0.1796
MA exact from n = 30 | EMA within 1% from n = 71


## Part 5 · Breaking a running sum

First Example 16-01.4 in 8-bit wraparound and saturating arithmetic, then a float32 recursive sum over a million samples.

In [6]:
x4 = [25, 30, 32, 33, 30]
s_wrap, mid = ds.running_sum(x4, 4, bits=8)
print("wraparound: intermediates", [int(v) for v in mid], "sums", [int(v) for v in s_wrap])
s_sat, acc = [], 0
for k, xn in enumerate(x4):
    m = max(-128, min(127, acc + xn))
    acc = max(-128, min(127, m - (x4[k - 4] if k >= 4 else 0)))
    s_sat.append(acc)
print("saturating:  sums", s_sat, "| true s[4] =", sum(x4[1:]))

# float32 recursive moving sum against the exact window sum
N, L = 16, 1_000_000
xf = np.random.default_rng(9).random(L).astype(np.float32)
acc = np.float32(0.0)
err = np.empty(L // 1000)
csum = np.concatenate([[0.0], np.cumsum(xf.astype(np.float64))])
for k in range(L):
    acc = np.float32(acc + xf[k])
    if k >= N:
        acc = np.float32(acc - xf[k - N])
    if k % 1000 == 999:
        exact = csum[k + 1] - csum[max(0, k + 1 - N)]
        err[k // 1000] = float(acc) - exact
fig, ax = plt.subplots(figsize=(6, 2.8))
ax.plot(np.arange(err.size) * 1000, err); ax.set_xlabel("sample"); ax.set_ylabel("float32 error")
plt.show()
print(f"float32 error after {L} samples: {err[-1]:.2e} (max |error| {np.max(np.abs(err)):.2e})")

wraparound: intermediates [25, 55, 87, 120, -106] sums [25, 55, 87, 120, 125]
saturating:  sums [25, 55, 87, 120, 102] | true s[4] = 125


float32 error after 1000000 samples: 1.04e-04 (max |error| 2.76e-04)


## Part 6 · The same moving sum in Verilog

`movsum4.v` adds and subtracts in 8 bits; the testbench compares it with a 32-bit reference model. Afterwards, edit the design to saturate and run it again: it should fail.

In [7]:
if shutil.which("iverilog"):
    subprocess.run(["iverilog", "-g2012", "-o", "_movsum4.vvp", "movsum4_tb.v", "movsum4.v"], check=True)
    out = subprocess.run(["vvp", "-n", "_movsum4.vvp"], capture_output=True, text=True).stdout
    print(out.strip())
    import os; os.remove("_movsum4.vvp")
else:
    print("iverilog not installed: skip (CI runs movsum4_tb.v through tools/run_hdl.py)")

intermediate wraps exercised: 264
TEST PASSED
movsum4_tb.v:53: $finish called at 20056000 (1ps)


**Discussion.** Answer the five questions in the [lab handout](README.md) before reading its answers.